In [33]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, year, month, round,countDistinct





In [ ]:
spark = SparkSession.builder \
    .appName("Seller Acquisition EDA") \
    .getOrCreate()

In [20]:
mql_df = spark.read.parquet(
    "../../data/silver/mql"
)

closed_deals_df = spark.read.parquet(
    "../../data/silver/closed_deals"
)

seller_acquisition_df = spark.read.parquet(
    "../../data/silver/seller_acquisition_staging"
)

sellers_df = spark.read.parquet(
    "../../data/silver/sellers"
)


orders_df = spark.read.parquet(
    "../../data/silver/orders"
)



order_delivery_df = spark.read.parquet(
    "../../data/silver/order_delivery_staging"
)

reviews_df = spark.read.parquet(
    "../../data/silver/reviews_staging"
)

seller_fulfillment_df = spark.read.parquet(
    "../../data/silver/seller_fulfillment_staging"
)

In [13]:
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)

print("MQL Count:",
      mql_df.count())

print("Closed Deals Count:",
      closed_deals_df.count())

print("Seller Acquisition Count:",
      seller_acquisition_df.count())

print("Seller Count:",
      sellers_df.count())


DATASET OVERVIEW
MQL Count: 8000
Closed Deals Count: 842
Seller Acquisition Count: 8000
Seller Count: 3095


In [14]:
mql_df.groupBy(
    "origin"
).count().orderBy(
    col("count").desc()
).show(
    truncate=False
)

+-----------------+-----+
|origin           |count|
+-----------------+-----+
|organic_search   |2296 |
|paid_search      |1586 |
|social           |1350 |
|unknown          |1099 |
|direct_traffic   |499  |
|email            |493  |
|referral         |284  |
|other            |150  |
|display          |118  |
|other_publicities|65   |
|NULL             |60   |
+-----------------+-----+



In [15]:
closed_deals_df.groupBy(
    "business_segment"
).count().orderBy(
    col("count").desc()
).show(
    truncate=False
)

+-------------------------------+-----+
|business_segment               |count|
+-------------------------------+-----+
|home_decor                     |105  |
|health_beauty                  |93   |
|car_accessories                |77   |
|household_utilities            |71   |
|construction_tools_house_garden|69   |
|audio_video_electronics        |64   |
|computers                      |34   |
|pet                            |30   |
|food_supplement                |28   |
|food_drink                     |26   |
|sports_leisure                 |25   |
|bed_bath_table                 |22   |
|bags_backpacks                 |22   |
|toys                           |20   |
|fashion_accessories            |19   |
|home_office_furniture          |14   |
|stationery                     |13   |
|phone_mobile                   |13   |
|small_appliances               |12   |
|handcrafted                    |12   |
+-------------------------------+-----+
only showing top 20 rows


In [21]:
from pyspark.sql.functions import min, max

orders_df.select(

    min("order_purchase_timestamp")
    .alias("first_order"),

    max("order_purchase_timestamp")
    .alias("last_order")

).show(truncate=False)

+-------------------+-------------------+
|first_order        |last_order         |
+-------------------+-------------------+
|2016-09-04 21:15:19|2018-10-17 17:30:18|
+-------------------+-------------------+



In [22]:
mql_df.select(

    min("first_contact_date")
    .alias("first_mql"),

    max("first_contact_date")
    .alias("last_mql")

).show(truncate=False)

+-------------------+-------------------+
|first_mql          |last_mql           |
+-------------------+-------------------+
|2017-06-14 00:00:00|2018-05-31 00:00:00|
+-------------------+-------------------+



In [23]:
closed_deals_df.select(

    min("won_date")
    .alias("first_conversion"),

    max("won_date")
    .alias("last_conversion")

).show(truncate=False)

+-------------------+-------------------+
|first_conversion   |last_conversion    |
+-------------------+-------------------+
|2017-12-05 02:00:00|2018-11-14 18:04:19|
+-------------------+-------------------+



In [29]:
from pyspark.sql.functions import (
    year,
    month,
    count
)

orders_monthly = (

    orders_df

    .groupBy(

        year("order_purchase_timestamp")
        .alias("year"),

        month("order_purchase_timestamp")
        .alias("month")

    )

    .agg(
        count("*").alias("orders")
    )

    .orderBy(
        "year",
        "month"
    )
)

orders_monthly.show(24)

+----+-----+------+
|year|month|orders|
+----+-----+------+
|2016|    9|     4|
|2016|   10|   320|
|2016|   12|     1|
|2017|    1|   799|
|2017|    2|  1777|
|2017|    3|  2681|
|2017|    4|  2380|
|2017|    5|  3684|
|2017|    6|  3240|
|2017|    7|  3994|
|2017|    8|  4328|
|2017|    9|  4272|
|2017|   10|  4631|
|2017|   11|  7542|
|2017|   12|  5668|
|2018|    1|  7244|
|2018|    2|  6716|
|2018|    3|  7209|
|2018|    4|  6546|
|2018|    5|  6805|
|2018|    6|  6041|
|2018|    7|  5723|
|2018|    8|  6426|
|2018|    9|    16|
+----+-----+------+
only showing top 24 rows


In [30]:
mql_monthly = (

    mql_df

    .groupBy(

        year("first_contact_date")
        .alias("year"),

        month("first_contact_date")
        .alias("month")

    )

    .agg(
        count("*").alias("mqls")
    )

    .orderBy(
        "year",
        "month"
    )
)

mql_monthly.show(50)

+----+-----+----+
|year|month|mqls|
+----+-----+----+
|2017|    6|   4|
|2017|    7| 239|
|2017|    8| 386|
|2017|    9| 312|
|2017|   10| 416|
|2017|   11| 445|
|2017|   12| 200|
|2018|    1|1141|
|2018|    2|1028|
|2018|    3|1174|
|2018|    4|1352|
|2018|    5|1303|
+----+-----+----+



In [31]:
closed_monthly = (

    closed_deals_df

    .groupBy(

        year("won_date")
        .alias("year"),

        month("won_date")
        .alias("month")

    )

    .agg(
        count("*").alias("closed_deals")
    )

    .orderBy(
        "year",
        "month"
    )
)

closed_monthly.show(50)

+----+-----+------------+
|year|month|closed_deals|
+----+-----+------------+
|2017|   12|           3|
|2018|    1|          73|
|2018|    2|         113|
|2018|    3|         147|
|2018|    4|         207|
|2018|    5|         122|
|2018|    6|          57|
|2018|    7|          37|
|2018|    8|          33|
|2018|    9|          23|
|2018|   10|          21|
|2018|   11|           6|
+----+-----+------------+



In [34]:
seller_acquisition_df.groupBy(
    "marketing_origin"
).agg(

    countDistinct("seller_id")
    .alias("seller_count")

).orderBy(
    col("seller_count").desc()
).show(
    truncate=False
)

+-----------------+------------+
|marketing_origin |seller_count|
+-----------------+------------+
|organic_search   |271         |
|paid_search      |195         |
|unknown          |179         |
|social           |75          |
|direct_traffic   |56          |
|referral         |24          |
|email            |15          |
|NULL             |14          |
|display          |6           |
|other            |4           |
|other_publicities|3           |
+-----------------+------------+



In [35]:
seller_acquisition_df.groupBy(
    "business_segment"
).count().orderBy(
    col("count").desc()
).show(
    truncate=False
)

+-------------------------------+-----+
|business_segment               |count|
+-------------------------------+-----+
|NULL                           |7159 |
|home_decor                     |105  |
|health_beauty                  |93   |
|car_accessories                |77   |
|household_utilities            |71   |
|construction_tools_house_garden|69   |
|audio_video_electronics        |64   |
|computers                      |34   |
|pet                            |30   |
|food_supplement                |28   |
|food_drink                     |26   |
|sports_leisure                 |25   |
|bed_bath_table                 |22   |
|bags_backpacks                 |22   |
|toys                           |20   |
|fashion_accessories            |19   |
|home_office_furniture          |14   |
|stationery                     |13   |
|phone_mobile                   |13   |
|small_appliances               |12   |
+-------------------------------+-----+
only showing top 20 rows


In [36]:
seller_acquisition_df.groupBy(
    "acquisition_risk_category"
).count().show(
    truncate=False
)

+-------------------------+-----+
|acquisition_risk_category|count|
+-------------------------+-----+
|Unknown                  |7158 |
|Standard                 |593  |
|High Risk                |249  |
+-------------------------+-----+



In [37]:
acquisition_perf = (
    seller_fulfillment_df
    .join(
        seller_acquisition_df.select(
            "seller_id",
            "marketing_origin"
        ),
        "seller_id",
        "inner"
    )
)

acquisition_perf.groupBy(
    "marketing_origin"
).agg(

    avg("seller_monthly_orders")
    .alias("avg_monthly_orders"),

    countDistinct("seller_id")
    .alias("seller_count")

).orderBy(
    col("avg_monthly_orders").desc()
).show(truncate=False)

+----------------+------------------+------------+
|marketing_origin|avg_monthly_orders|seller_count|
+----------------+------------------+------------+
|unknown         |57.75218658892128 |81          |
|other           |29.412371134020617|2           |
|paid_search     |26.35082872928177 |101         |
|social          |12.995495495495495|31          |
|organic_search  |12.86032689450223 |113         |
|referral        |4.298701298701299 |9           |
|direct_traffic  |4.2110091743119265|31          |
|email           |3.5               |6           |
|NULL            |1.3636363636363635|4           |
|display         |1.2857142857142858|2           |
+----------------+------------------+------------+



In [38]:
acquisition_perf.groupBy(
    "marketing_origin",
    "workload_bucket"
).count().show(
    truncate=False
)

+----------------+---------------+-----+
|marketing_origin|workload_bucket|count|
+----------------+---------------+-----+
|NULL            |Low Volume     |11   |
|social          |Low Volume     |321  |
|unknown         |High Volume    |424  |
|display         |Low Volume     |7    |
|paid_search     |Medium Volume  |526  |
|organic_search  |Medium Volume  |207  |
|paid_search     |Low Volume     |922  |
|direct_traffic  |Low Volume     |218  |
|social          |Medium Volume  |123  |
|unknown         |Low Volume     |794  |
|unknown         |Medium Volume  |154  |
|organic_search  |Low Volume     |1139 |
|other           |Medium Volume  |70   |
|email           |Low Volume     |24   |
|referral        |Low Volume     |77   |
|other           |Low Volume     |27   |
+----------------+---------------+-----+



In [39]:
acquisition_delivery_df = (
    order_delivery_df.alias("od")

    .join(
        seller_acquisition_df.alias("sa"),

        col("od.primary_seller_id")
        ==
        col("sa.seller_id"),

        "inner"
    )
)

In [40]:
acquisition_delivery_df.groupBy(
    "marketing_origin"
).agg(

    round(
        avg("delay_days"),
        2
    ).alias("avg_delay_days"),

    count("*")
    .alias("order_count")

).orderBy(
    col("avg_delay_days").desc()
).show(
    truncate=False
)

+----------------+--------------+-----------+
|marketing_origin|avg_delay_days|order_count|
+----------------+--------------+-----------+
|other           |-8.42         |89         |
|email           |-10.77        |22         |
|unknown         |-11.23        |1261       |
|social          |-11.55        |389        |
|paid_search     |-11.64        |1225       |
|NULL            |-11.73        |11         |
|direct_traffic  |-11.85        |192        |
|organic_search  |-12.41        |1194       |
|referral        |-12.64        |69         |
|display         |-13.43        |7          |
+----------------+--------------+-----------+



In [41]:
acquisition_delivery_df.groupBy(
    "marketing_origin"
).agg(

    round(
        avg(
            when(
                col("delay_days") <= 0,
                1
            ).otherwise(0)
        ) * 100,
        2
    ).alias("on_time_rate"),

    count("*")
    .alias("order_count")

).orderBy(
    col("on_time_rate").desc()
).show(
    truncate=False
)

+----------------+------------+-----------+
|marketing_origin|on_time_rate|order_count|
+----------------+------------+-----------+
|display         |100.0       |7          |
|referral        |97.1        |69         |
|organic_search  |95.48       |1194       |
|email           |95.45       |22         |
|social          |94.09       |389        |
|paid_search     |93.71       |1225       |
|direct_traffic  |92.71       |192        |
|NULL            |90.91       |11         |
|unknown         |90.8        |1261       |
|other           |85.39       |89         |
+----------------+------------+-----------+



In [42]:
acquisition_delivery_df.groupBy(

    "marketing_origin",

    "delivery_status_category"

).count().orderBy(

    "marketing_origin"

).show(
    truncate=False
)

+----------------+------------------------+-----+
|marketing_origin|delivery_status_category|count|
+----------------+------------------------+-----+
|NULL            |Early                   |10   |
|NULL            |Late                    |1    |
|direct_traffic  |Early                   |175  |
|direct_traffic  |On Time                 |3    |
|direct_traffic  |Late                    |12   |
|direct_traffic  |Unknown                 |2    |
|display         |Early                   |6    |
|display         |On Time                 |1    |
|email           |Early                   |21   |
|email           |Late                    |1    |
|organic_search  |On Time                 |21   |
|organic_search  |Late                    |37   |
|organic_search  |Early                   |1119 |
|organic_search  |Unknown                 |17   |
|other           |Late                    |10   |
|other           |Early                   |75   |
|other           |On Time                 |1    |


In [43]:
acquisition_delivery_df.groupBy(

    "marketing_origin",

    "distance_bucket"

).count().show(
    truncate=False
)

+----------------+---------------+-----+
|marketing_origin|distance_bucket|count|
+----------------+---------------+-----+
|organic_search  |Cross Region   |448  |
|unknown         |Same Region    |358  |
|NULL            |Same State     |7    |
|social          |Same State     |164  |
|email           |Cross Region   |5    |
|other           |Same State     |47   |
|other           |Same Region    |10   |
|referral        |Same Region    |19   |
|social          |Cross Region   |114  |
|organic_search  |Same Region    |253  |
|paid_search     |Same Region    |335  |
|paid_search     |Cross Region   |372  |
|display         |Same Region    |1    |
|NULL            |Cross Region   |4    |
|unknown         |Cross Region   |398  |
|social          |Same Region    |111  |
|direct_traffic  |Cross Region   |74   |
|email           |Same State     |13   |
|referral        |Same State     |19   |
|display         |Same State     |2    |
+----------------+---------------+-----+
only showing top

In [44]:
acquisition_delivery_df.groupBy(
    "marketing_origin"
).agg(

    round(
        avg(
            when(
                col("is_multi_seller_order"),
                1
            ).otherwise(0)
        ) * 100,
        2
    ).alias("multi_seller_rate")

).show(
    truncate=False
)

+----------------+-----------------+
|marketing_origin|multi_seller_rate|
+----------------+-----------------+
|unknown         |1.43             |
|NULL            |0.0              |
|display         |0.0              |
|referral        |1.45             |
|other           |0.0              |
|email           |0.0              |
|social          |0.77             |
|direct_traffic  |2.08             |
|organic_search  |1.34             |
|paid_search     |1.55             |
+----------------+-----------------+



In [45]:
acquisition_reviews_df = (

    reviews_df.alias("rv")

    .join(

        seller_acquisition_df.alias("sa"),

        col("rv.primary_seller_id")
        ==
        col("sa.seller_id"),

        "inner"
    )
)

In [46]:
acquisition_reviews_df.groupBy(
    "marketing_origin"
).agg(

    round(
        avg("review_score"),
        2
    ).alias("avg_review_score"),

    count("*")
    .alias("review_count")

).orderBy(
    col("avg_review_score").desc()
).show(
    truncate=False
)

+----------------+----------------+------------+
|marketing_origin|avg_review_score|review_count|
+----------------+----------------+------------+
|email           |4.48            |21          |
|organic_search  |4.39            |1189        |
|paid_search     |4.32            |1222        |
|referral        |4.3             |69          |
|social          |4.28            |389         |
|unknown         |4.17            |1252        |
|direct_traffic  |4.14            |191         |
|NULL            |4.09            |11          |
|other           |3.72            |89          |
|display         |3.71            |7           |
+----------------+----------------+------------+



In [47]:
acquisition_reviews_df.groupBy(
    "marketing_origin"
).agg(

    round(
        avg(
            when(
                col("review_score") <= 2,
                1
            ).otherwise(0)
        ) * 100,
        2
    ).alias("negative_review_rate"),

    count("*")
    .alias("review_count")

).orderBy(
    col("negative_review_rate").desc()
).show(
    truncate=False
)

+----------------+--------------------+------------+
|marketing_origin|negative_review_rate|review_count|
+----------------+--------------------+------------+
|display         |28.57               |7           |
|other           |24.72               |89          |
|NULL            |18.18               |11          |
|unknown         |13.58               |1252        |
|direct_traffic  |12.57               |191         |
|social          |10.8                |389         |
|paid_search     |10.31               |1222        |
|referral        |10.14               |69          |
|email           |9.52                |21          |
|organic_search  |8.83                |1189        |
+----------------+--------------------+------------+



In [48]:
acquisition_reviews_df.groupBy(
    "marketing_origin"
).agg(

    round(
        avg(
            when(
                col("delayed_delivery_review_flag"),
                1
            ).otherwise(0)
        ) * 100,
        2
    ).alias(
        "delivery_complaint_rate"
    ),

    count("*")
    .alias("review_count")

).orderBy(
    col("delivery_complaint_rate").desc()
).show(
    truncate=False
)

+----------------+-----------------------+------------+
|marketing_origin|delivery_complaint_rate|review_count|
+----------------+-----------------------+------------+
|unknown         |5.35                   |1252        |
|other           |4.49                   |89          |
|social          |3.34                   |389         |
|direct_traffic  |3.14                   |191         |
|paid_search     |2.86                   |1222        |
|organic_search  |1.43                   |1189        |
|NULL            |0.0                    |11          |
|display         |0.0                    |7           |
|referral        |0.0                    |69          |
|email           |0.0                    |21          |
+----------------+-----------------------+------------+



In [49]:
acquisition_reviews_df.groupBy(

    "marketing_origin",

    "sentiment_category"

).count().orderBy(

    "marketing_origin"

).show(
    truncate=False
)

+----------------+------------------+-----+
|marketing_origin|sentiment_category|count|
+----------------+------------------+-----+
|NULL            |Positive          |9    |
|NULL            |Negative          |2    |
|direct_traffic  |Neutral           |19   |
|direct_traffic  |Positive          |148  |
|direct_traffic  |Negative          |24   |
|display         |Negative          |2    |
|display         |Positive          |5    |
|email           |Positive          |19   |
|email           |Negative          |2    |
|organic_search  |Neutral           |65   |
|organic_search  |Negative          |105  |
|organic_search  |Positive          |1019 |
|other           |Neutral           |8    |
|other           |Positive          |59   |
|other           |Negative          |22   |
|paid_search     |Negative          |126  |
|paid_search     |Positive          |1017 |
|paid_search     |Neutral           |79   |
|referral        |Negative          |7    |
|referral        |Neutral       

In [50]:
customer_satisfaction_summary = (

    acquisition_reviews_df

    .groupBy(
        "marketing_origin"
    )

    .agg(

        round(
            avg("review_score"),
            2
        ).alias(
            "avg_review_score"
        ),

        round(
            avg(
                when(
                    col("review_score") <= 2,
                    1
                ).otherwise(0)
            ) * 100,
            2
        ).alias(
            "negative_review_rate"
        ),

        round(
            avg(
                when(
                    col("review_score") >= 4,
                    1
                ).otherwise(0)
            ) * 100,
            2
        ).alias(
            "positive_review_rate"
        ),

        count("*")
        .alias(
            "review_count"
        )
    )

    .orderBy(
        col("avg_review_score").desc()
    )
)

customer_satisfaction_summary.show(
    truncate=False
)

+----------------+----------------+--------------------+--------------------+------------+
|marketing_origin|avg_review_score|negative_review_rate|positive_review_rate|review_count|
+----------------+----------------+--------------------+--------------------+------------+
|email           |4.48            |9.52                |90.48               |21          |
|organic_search  |4.39            |8.83                |85.7                |1189        |
|paid_search     |4.32            |10.31               |83.22               |1222        |
|referral        |4.3             |10.14               |84.06               |69          |
|social          |4.28            |10.8                |82.01               |389         |
|unknown         |4.17            |13.58               |79.71               |1252        |
|direct_traffic  |4.14            |12.57               |77.49               |191         |
|NULL            |4.09            |18.18               |81.82               |11          |